In [ ]:
# ==============================================================
# EXERCISE 8: SELF-ORGANIZING MAP (SOM)
# Customer Segmentation and Cluster Exploration System
# Dataset: Mall Customers
# ==============================================================

import numpy as np
import pandas as pd
import warnings
import os

warnings.filterwarnings("ignore")

from sklearn.preprocessing import MinMaxScaler

# ==============================================================
# 1. LOAD DATASET - MULTIPLE SOURCES + UPLOAD FALLBACK
# ==============================================================

def load_dataset():
    urls = [
        "https://raw.githubusercontent.com/piyush-taneja/Mall-Customers/master/Mall_Customers.csv",
        "https://raw.githubusercontent.com/tanishq21/Mall-Customers/main/Mall_Customers.csv",
        "https://raw.githubusercontent.com/tugrulhkarabulut/K-Means-Clustering/master/example_datasets/Mall_Customers.csv",
        "https://raw.githubusercontent.com/tirthajyoti/Machine-Learning-with-Python/master/Datasets/Mall_Customers.csv",
        "https://raw.githubusercontent.com/abcom-mltutorials/mall/main/Mall_Customers.csv"
    ]

    required = [
        "Age",
        "Annual Income (k$)",
        "Spending Score (1-100)"
    ]

    # First try the local file, if it already exists.
    local_paths = [
        "/content/Mall_Customers.csv",
        "Mall_Customers.csv"
    ]

    for path in local_paths:
        if os.path.exists(path):
            try:
                data = pd.read_csv(path)
                data.columns = data.columns.str.strip()

                if all(c in data.columns for c in required):
                    if "CustomerID" not in data.columns:
                        data.insert(0, "CustomerID", range(1, len(data) + 1))

                    print("Dataset loaded from local file:", path)
                    return data
            except Exception:
                pass

    # Try downloading from multiple repositories.
    for url in urls:
        try:
            print("Trying dataset source...")
            data = pd.read_csv(url, timeout=15)
            data.columns = data.columns.str.strip()

            if all(c in data.columns for c in required):
                if "CustomerID" not in data.columns:
                    data.insert(0, "CustomerID", range(1, len(data) + 1))

                print("Dataset downloaded successfully!")
                return data

        except Exception as error:
            print("Source unavailable; trying the next source.")

    # If running in Google Colab, allow the user to upload the CSV.
    try:
        from google.colab import files

        print("\nAutomatic download failed.")
        print("Please upload the original Mall_Customers.csv file.")
        uploaded = files.upload()

        for filename in uploaded:
            try:
                data = pd.read_csv(filename)
                data.columns = data.columns.str.strip()

                if all(c in data.columns for c in required):
                    if "CustomerID" not in data.columns:
                        data.insert(
                            0, "CustomerID", range(1, len(data) + 1)
                        )

                    print("Uploaded dataset loaded successfully!")
                    return data

            except Exception:
                pass

    except ImportError:
        pass

    raise RuntimeError(
        "Unable to load the dataset. Please download Mall_Customers.csv "
        "and upload it to Google Colab."
    )


df = load_dataset()

FEATURES = [
    "Age",
    "Annual Income (k$)",
    "Spending Score (1-100)"
]

df = df.dropna(subset=FEATURES).reset_index(drop=True)

X_raw = df[FEATURES].values.astype(float)
customer_ids = df["CustomerID"].values

scaler = MinMaxScaler()
X = scaler.fit_transform(X_raw)

print("\n" + "=" * 55)
print("MALL CUSTOMERS DATASET LOADED")
print("=" * 55)
print("Total Customers:", len(df))
print("Number of Features:", len(FEATURES))
print("Features:", FEATURES)
print("\nFirst Five Customers:")
print(df[["CustomerID"] + FEATURES].head().to_string(index=False))


# ==============================================================
# 2. SELF-ORGANIZING MAP MODEL
# ==============================================================

class SOM:

    def __init__(
        self,
        n_neurons=4,
        learning_rate=0.5,
        iterations=1000,
        random_state=42
    ):
        self.n_neurons = n_neurons
        self.learning_rate = learning_rate
        self.iterations = iterations
        self.random_state = random_state
        self.weights = None

    def find_bmu(self, sample):
        distances = np.linalg.norm(
            self.weights - sample,
            axis=1
        )

        bmu = int(np.argmin(distances))
        return bmu, distances

    def fit(self, data):
        rng = np.random.default_rng(self.random_state)

        selected = rng.choice(
            len(data),
            size=self.n_neurons,
            replace=False
        )

        self.weights = data[selected].copy()

        for iteration in range(self.iterations):

            sample = data[rng.integers(len(data))]

            bmu, _ = self.find_bmu(sample)

            progress = iteration / max(self.iterations - 1, 1)

            lr = self.learning_rate * (1 - progress)

            initial_radius = max(1.0, self.n_neurons / 2)

            radius = max(
                0.5,
                initial_radius * (1 - progress)
            )

            # Competitive learning and neighborhood updates.
            for neuron in range(self.n_neurons):

                grid_distance = abs(neuron - bmu)

                neighborhood = np.exp(
                    -(grid_distance ** 2) / (2 * radius ** 2)
                )

                self.weights[neuron] += (
                    lr
                    * neighborhood
                    * (sample - self.weights[neuron])
                )

        return self

    def predict(self, data):
        labels = []

        for sample in data:
            bmu, _ = self.find_bmu(sample)
            labels.append(bmu)

        return np.array(labels)

    def quantization_error(self, data):
        distances = []

        for sample in data:
            bmu, all_distances = self.find_bmu(sample)
            distances.append(all_distances[bmu])

        return float(np.mean(distances))


def train_som(neurons=4, learning_rate=0.5, iterations=1000):
    return SOM(
        n_neurons=neurons,
        learning_rate=learning_rate,
        iterations=iterations
    ).fit(X)


def cluster_counts(som):
    labels = som.predict(X)

    return np.bincount(
        labels,
        minlength=som.n_neurons
    )


def display_clusters(som):
    labels = som.predict(X)

    for cluster in range(som.n_neurons):

        members = np.where(labels == cluster)[0]

        print(
            f"\nCluster {cluster + 1}: "
            f"{len(members)} customers"
        )

        if len(members) == 0:
            print("No customers assigned.")
            continue

        result = df.iloc[members][
            ["CustomerID"] + FEATURES
        ]

        print(result.to_string(index=False))


# ==============================================================
# MENU 1: CUSTOMER SEGMENTATION USING SOM
# ==============================================================

def customer_segmentation():

    print("\n--- CUSTOMER SEGMENTATION USING SOM ---")

    try:
        neurons = int(input("Enter number of SOM neurons: "))

        iterations = int(
            input("Enter training iterations (default 1000): ") or "1000"
        )

        if not 2 <= neurons <= len(X) or iterations < 1:
            print("Invalid neuron count or iteration count.")
            return

        som = train_som(
            neurons=neurons,
            iterations=iterations
        )

        display_clusters(som)

        print(
            f"\nQuantization Error: "
            f"{som.quantization_error(X):.6f}"
        )

        prototypes = scaler.inverse_transform(som.weights)

        print("\nCluster Prototype Values:")

        print(
            pd.DataFrame(
                prototypes,
                columns=FEATURES,
                index=[
                    f"Cluster {i + 1}"
                    for i in range(neurons)
                ]
            ).round(2)
        )

    except ValueError:
        print("Please enter valid integers.")


# ==============================================================
# MENU 2: BEST MATCHING UNIT EXPLORER
# ==============================================================

def bmu_explorer():

    print("\n--- BEST MATCHING UNIT EXPLORER ---")

    try:
        index = int(
            input(f"Enter customer row index (0-{len(df)-1}): ")
        )

        neurons = int(input("Enter number of SOM neurons: "))

        if not 0 <= index < len(df):
            print("Invalid customer index.")
            return

        if not 2 <= neurons <= len(X):
            print("Invalid neuron count.")
            return

        som = train_som(neurons=neurons)

        sample = X[index]

        bmu, distances = som.find_bmu(sample)

        print("\nCustomer ID:", customer_ids[index])

        print("\nCustomer Features:")

        for feature, value in zip(FEATURES, X_raw[index]):
            print(f"{feature}: {value}")

        print("\nDistance from Each Neuron:")

        for neuron, distance in enumerate(distances):
            print(
                f"Neuron {neuron + 1}: {distance:.6f}"
            )

        print("\nBest Matching Unit:", bmu + 1)
        print("Assigned Cluster:", bmu + 1)
        print(f"BMU Distance: {distances[bmu]:.6f}")

    except ValueError:
        print("Please enter valid numbers.")


# ==============================================================
# MENU 3: DYNAMIC CLUSTER COUNT CHALLENGE
# ==============================================================

def dynamic_cluster_challenge():

    print("\n--- DYNAMIC CLUSTER COUNT CHALLENGE ---")

    results = []

    for neurons in range(2, 7):

        som = train_som(neurons=neurons)

        counts = cluster_counts(som)

        empty = int(np.sum(counts == 0))
        largest = int(np.max(counts))

        nonempty = counts[counts > 0]

        smallest = (
            int(np.min(nonempty))
            if len(nonempty) > 0 else 0
        )

        qe = som.quantization_error(X)

        print(f"\nNumber of Neurons: {neurons}")
        print("Customers per Cluster:", counts.tolist())
        print("Empty Clusters:", empty)
        print("Largest Cluster:", largest)
        print("Smallest Non-empty Cluster:", smallest)
        print(f"Quantization Error: {qe:.6f}")

        if largest > 0.6 * len(X):
            print("Warning: Highly imbalanced cluster distribution.")

        results.append({
            "Neurons": neurons,
            "Empty Clusters": empty,
            "Quantization Error": qe,
            "Largest Cluster": largest,
            "Smallest Non-empty Cluster": smallest
        })

    report = pd.DataFrame(results)

    print("\n--- CLUSTER COUNT COMPARISON ---")
    print(report.to_string(index=False))

    suitable = report.sort_values(
        ["Empty Clusters", "Quantization Error"]
    ).iloc[0]

    print(
        "\nSuggested Number of Neurons:",
        int(suitable["Neurons"])
    )


# ==============================================================
# MENU 4: CUSTOMER CLUSTER MIGRATION ANALYZER
# ==============================================================

def cluster_migration():

    print("\n--- CUSTOMER CLUSTER MIGRATION ANALYZER ---")

    try:
        index = int(
            input(f"Enter customer row index (0-{len(df)-1}): ")
        )

        neurons = int(input("Enter number of SOM neurons: "))

        if not 0 <= index < len(df):
            print("Invalid customer index.")
            return

        if not 2 <= neurons <= len(X):
            print("Invalid neuron count.")
            return

        som = train_som(neurons=neurons)

        original = X_raw[index].copy()

        print("\n1. Annual Income")
        print("2. Spending Score")

        choice = input("Choose a feature to modify: ").strip()

        if choice == "1":
            feature_index = 1
        elif choice == "2":
            feature_index = 2
        else:
            print("Invalid choice.")
            return

        scaled_original = scaler.transform(
            original.reshape(1, -1)
        )[0]

        original_bmu, _ = som.find_bmu(scaled_original)

        print("\nCustomer ID:", customer_ids[index])
        print("Original Cluster:", original_bmu + 1)

        lower = max(0, original[feature_index] - 40)
        upper = original[feature_index] + 40

        values = np.linspace(lower, upper, 81)

        previous_cluster = original_bmu
        migrations = []

        print("\nModified Value | Assigned Cluster")

        for value in values:

            modified = original.copy()
            modified[feature_index] = value

            scaled = scaler.transform(
                modified.reshape(1, -1)
            )[0]

            cluster, _ = som.find_bmu(scaled)

            print(f"{value:8.2f} | Cluster {cluster + 1}")

            if cluster != previous_cluster:
                migrations.append((value, cluster + 1))
                previous_cluster = cluster

        if migrations:

            print("\nDetected Cluster Migrations:")

            for value, cluster in migrations:
                print(
                    f"Feature value {value:.2f} "
                    f"-> Cluster {cluster}"
                )

        else:
            print("\nNo cluster migration detected in this range.")

    except ValueError:
        print("Please enter valid numbers.")


# ==============================================================
# MENU 5: CLUSTER PROTOTYPE GENERATOR
# ==============================================================

def cluster_prototypes():

    print("\n--- CLUSTER PROTOTYPE GENERATOR ---")

    try:
        neurons = int(input("Enter number of SOM neurons: "))

        if not 2 <= neurons <= len(X):
            print("Invalid neuron count.")
            return

        som = train_som(neurons=neurons)

        labels = som.predict(X)

        prototypes = scaler.inverse_transform(som.weights)

        for i, prototype in enumerate(prototypes):

            age, income, spending = prototype

            count = int(np.sum(labels == i))

            print(f"\nCluster {i + 1}")
            print("Customers:", count)
            print(f"Representative Age: {age:.2f}")
            print(f"Annual Income: {income:.2f} k$")
            print(f"Spending Score: {spending:.2f}")

            age_description = (
                "Young" if age < 35
                else "Middle-Aged" if age < 55
                else "Older"
            )

            income_description = (
                "Low-Income" if income < 40
                else "Moderate-Income" if income < 70
                else "High-Income"
            )

            spending_description = (
                "Low-Spending" if spending < 40
                else "Average-Spending" if spending < 60
                else "High-Spending"
            )

            print(
                f"Description: {age_description} Customers, "
                f"{income_description}, {spending_description}"
            )

    except ValueError:
        print("Please enter a valid integer.")


# ==============================================================
# MENU 6: OUTLIER CUSTOMER DETECTOR
# ==============================================================

def outlier_detector():

    print("\n--- OUTLIER CUSTOMER DETECTOR ---")

    try:
        neurons = int(input("Enter number of SOM neurons: "))

        if not 2 <= neurons <= len(X):
            print("Invalid neuron count.")
            return

        som = train_som(neurons=neurons)

        labels = som.predict(X)

        distances = np.array([
            som.find_bmu(sample)[1][labels[i]]
            for i, sample in enumerate(X)
        ])

        threshold = np.percentile(distances, 95)

        outlier_indices = np.where(distances >= threshold)[0]

        ranked = sorted(
            outlier_indices,
            key=lambda i: distances[i],
            reverse=True
        )[:5]

        print(f"\nOutlier Threshold (95th percentile): {threshold:.6f}")
        print("\nTop 5 Most Unusual Customers:")

        records = []

        for i in ranked:

            records.append({
                "Customer ID": customer_ids[i],
                "Age": X_raw[i, 0],
                "Annual Income": X_raw[i, 1],
                "Spending Score": X_raw[i, 2],
                "Cluster": labels[i] + 1,
                "BMU Distance": distances[i]
            })

        print(pd.DataFrame(records).to_string(index=False))

    except ValueError:
        print("Please enter a valid integer.")


# ==============================================================
# MENU 7: SOM CLUSTER QUALITY COMPARISON
# ==============================================================

def cluster_quality_comparison():

    print("\n--- SOM CLUSTER QUALITY COMPARISON ---")

    results = []

    for neurons in range(2, 7):

        for learning_rate in [0.1, 0.3, 0.5]:

            iterations = 1000

            som = train_som(
                neurons=neurons,
                learning_rate=learning_rate,
                iterations=iterations
            )

            labels = som.predict(X)

            counts = np.bincount(
                labels,
                minlength=neurons
            )

            results.append({
                "Neurons": neurons,
                "Learning Rate": learning_rate,
                "Iterations": iterations,
                "Empty Clusters": int(np.sum(counts == 0)),
                "Quantization Error": som.quantization_error(X),
                "Largest Cluster": int(np.max(counts)),
                "Smallest Cluster": int(np.min(counts))
            })

    report = pd.DataFrame(results).sort_values(
        ["Quantization Error", "Empty Clusters"]
    )

    print(
        report.to_string(
            index=False,
            float_format=lambda value: f"{value:.4f}"
        )
    )

    best = report.iloc[0]

    print("\nBest Configuration:")
    print("Neurons:", int(best["Neurons"]))
    print("Learning Rate:", best["Learning Rate"])
    print(
        f"Quantization Error: "
        f"{best['Quantization Error']:.6f}"
    )


# ==============================================================
# MAIN MENU
# ==============================================================

def main():

    while True:

        print("\n" + "=" * 65)
        print(" SOM CUSTOMER SEGMENTATION AND CLUSTER EXPLORATION")
        print("=" * 65)

        print("1. Customer Segmentation using SOM")
        print("2. Best Matching Unit Explorer")
        print("3. Dynamic Cluster Count Challenge")
        print("4. Customer Cluster Migration Analyzer")
        print("5. Cluster Prototype Generator")
        print("6. Outlier Customer Detector")
        print("7. SOM Cluster Quality Comparison")
        print("8. Exit")

        choice = input("\nEnter your choice (1-8): ").strip()

        try:

            if choice == "1":
                customer_segmentation()

            elif choice == "2":
                bmu_explorer()

            elif choice == "3":
                dynamic_cluster_challenge()

            elif choice == "4":
                cluster_migration()

            elif choice == "5":
                cluster_prototypes()

            elif choice == "6":
                outlier_detector()

            elif choice == "7":
                cluster_quality_comparison()

            elif choice == "8":
                print("Exiting SOM Customer Segmentation System.")
                break

            else:
                print("Invalid choice. Please select 1-8.")

        except Exception as error:
            print("Error:", error)


main()

Trying dataset source...
Source unavailable; trying the next source.
Trying dataset source...
Source unavailable; trying the next source.
Trying dataset source...
Source unavailable; trying the next source.
Trying dataset source...
Source unavailable; trying the next source.
Trying dataset source...
Source unavailable; trying the next source.

Automatic download failed.
Please upload the original Mall_Customers.csv file.


Saving archive.zip to archive.zip
Uploaded dataset loaded successfully!

MALL CUSTOMERS DATASET LOADED
Total Customers: 200
Number of Features: 3
Features: ['Age', 'Annual Income (k$)', 'Spending Score (1-100)']

First Five Customers:
 CustomerID  Age  Annual Income (k$)  Spending Score (1-100)
          1   19                  15                      39
          2   21                  15                      81
          3   20                  16                       6
          4   23                  16                      77
          5   31                  17                      40

 SOM CUSTOMER SEGMENTATION AND CLUSTER EXPLORATION
1. Customer Segmentation using SOM
2. Best Matching Unit Explorer
3. Dynamic Cluster Count Challenge
4. Customer Cluster Migration Analyzer
5. Cluster Prototype Generator
6. Outlier Customer Detector
7. SOM Cluster Quality Comparison
8. Exit

Enter your choice (1-8): 8
Exiting SOM Customer Segmentation System.
